## Example 1: Two replicas edit text concurrently and converge

In [ ]:
import importlib.metadata

from loro import ExportMode, LoroDoc

# Example 1: each peer edits its own copy, then they exchange updates; both end up identical
assert importlib.metadata.version("loro") == "1.17.1"
a, b = LoroDoc(), LoroDoc()
a.get_text("t").insert(0, "Hello world"); a.commit()
b.import_(a.export(ExportMode.Snapshot()))
a.get_text("t").insert(5, ","); a.commit()
b.get_text("t").insert(11, "!"); b.commit()
a.import_(b.export(ExportMode.Updates(a.oplog_vv)))
b.import_(a.export(ExportMode.Updates(b.oplog_vv)))
assert a.get_text("t").to_string() == b.get_text("t").to_string() == "Hello, world!", a.get_text("t").to_string()
print("Example 1 OK:", a.get_text("t").to_string())

## Example 2: Maps, lists and nested containers

In [ ]:
from loro import LoroDoc, LoroList, LoroMap

# Example 2: JSON-like data whose concurrent edits merge per key and per element
doc = LoroDoc()
cfg = doc.get_map("config")
cfg.insert("name", "run-7")
cfg.insert("epochs", 3)
tags = cfg.insert_container("tags", LoroList())
tags.push("gpu"); tags.push("fp8")
doc.commit()
value = doc.get_deep_value()
assert value == {"config": {"name": "run-7", "epochs": 3, "tags": ["gpu", "fp8"]}}, value
print("Example 2 OK:", value)

## Example 3: Counters add up across peers

In [ ]:
from loro import ExportMode, LoroDoc

# Example 3: a CRDT counter: concurrent increments are summed, never lost
a, b = LoroDoc(), LoroDoc()
a.get_counter("visits").increment(2); a.commit()
b.get_counter("visits").increment(5); b.commit()
a.import_(b.export(ExportMode.Snapshot()))
b.import_(a.export(ExportMode.Snapshot()))
assert a.get_counter("visits").value == b.get_counter("visits").value == 7, a.get_counter("visits").value
print("Example 3 OK:", a.get_counter("visits").value)

## Example 4: Time travel: check out an earlier version

In [ ]:
from loro import LoroDoc

# Example 4: every commit is kept; checkout() shows the document as it was, checkout_to_latest() comes back
doc = LoroDoc()
text = doc.get_text("t")
text.insert(0, "v1"); doc.commit()
before = doc.state_frontiers
text.delete(0, 2); text.insert(0, "v2"); doc.commit()
doc.checkout(before)
assert doc.get_text("t").to_string() == "v1"
doc.checkout_to_latest()
assert doc.get_text("t").to_string() == "v2"
print("Example 4 OK: v1 -> v2, history kept")

## Example 5: Undo and a movable tree

In [ ]:
from loro import LoroDoc, UndoManager

# Example 5: undo reverts local edits; a tree holds nodes that can be moved under new parents
doc = LoroDoc()
undo = UndoManager(doc)
doc.get_text("t").insert(0, "draft"); doc.commit()
assert undo.undo() and doc.get_text("t").to_string() == ""
tree = doc.get_tree("outline")
root = tree.create(); child = tree.create(root); other = tree.create()
tree.mov(child, other)
doc.commit()
assert tree.parent(child) == other and tree.children(root) in ([], None), (tree.parent(child), tree.children(root))
print("Example 5 OK")